# Notebook 03: TrOCR Handwritten Extraction

This notebook loads the TrOCR handwritten OCR model, runs it on preprocessed page images produced by the preprocessing step, saves OCR output, and optionally tests accuracy against a small ground-truth file.

In [ ]:
from pathlib import Path
import json
import sys


REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
from src.ocr import TrOCRExtractor, evaluate_accuracy

MODEL_NAME = "microsoft/trocr-base-handwritten"
CACHE_DIR = REPO_ROOT / "models" / "trocr"
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_dirs = [
    DATA_DIR / "preprocessed_images",
    DATA_DIR / "processed_images",
    REPO_ROOT / "outputs" / "preprocessed_images",
]

image_dir = next((path for path in image_dirs if path.exists()), DATA_DIR / "preprocessed_images")
image_dir.mkdir(parents=True, exist_ok=True)

print(f"Using image directory: {image_dir}")
print(f"Model cache directory: {CACHE_DIR}")

In [ ]:
from PIL import Image

extractor = TrOCRExtractor(model_name=MODEL_NAME, cache_dir=CACHE_DIR)

image_patterns = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff", "*.webp"]
image_paths = []
for pattern in image_patterns:
    image_paths.extend(image_dir.glob(pattern))

image_paths = sorted(set(image_paths))

if not image_paths:
    raise FileNotFoundError(
        f"No preprocessed images were found in {image_dir}. Place your processed page images there and re-run this cell."
    )

print(f"Found {len(image_paths)} image(s) for OCR")
print("Sample files:", [path.name for path in image_paths[:5]])

In [ ]:
results = []
for image_path in image_paths:
    text = extractor.extract_text_from_image(image_path)
    results.append({"image": image_path.name, "text": text})

results_path = OUTPUT_DIR / "trocr_results.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Saved OCR results to {results_path}")

for item in results[:5]:
    print("-" * 60)
    print(item["image"])
    print(item["text"][:400])

In [ ]:
ground_truth_path = DATA_DIR / "ground_truth.json"
if ground_truth_path.exists():
    ground_truth = json.loads(ground_truth_path.read_text(encoding="utf-8"))
    metrics = evaluate_accuracy(results, ground_truth)
    print("Accuracy metrics:", metrics)
else:
    print("No ground_truth.json file found. Create one with keys like {'page_1.png': 'expected text'} to test accuracy.")